Copyright Amazon.com, Inc. or its affiliates. All Rights Reserved. SPDX-License-Identifier: Apache-2.0

# Salesforce Lightning Platform as AgentCore Gateway Target

## Overview

This notebook walks through adding **Salesforce Lightning Platform** as an integration target on [Amazon Bedrock AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html) using OAuth2 (`client_credentials` flow). Once configured, the gateway exposes 43 Salesforce tools (Account, Case, Contact, Lead, Opportunity, Campaign, etc.) as MCP tools that any agent can invoke.

| Information | Details |
|:---|:---|
| Target type | Integration Provider Template (Salesforce Lightning Platform) |
| Outbound auth | CustomOauth2 (Salesforce Connected App, client_credentials) |
| Inbound auth | Amazon Cognito (M2M) |
| Tools exposed | 43 (CRUD on Account, Case, Contact, Lead, Opportunity, Campaign, etc.) |
| Salesforce API version | v62.0 |

### Prerequisites

Before starting this notebook, you need:

1. **AWS account** with access to Amazon Bedrock AgentCore
2. **Salesforce Developer Edition org** — free at [developer.salesforce.com/signup](https://developer.salesforce.com/signup)
3. **Salesforce Connected App** configured for OAuth2 `client_credentials` flow (see Step 1 below)
4. **Python 3.11+** with Jupyter kernel
5. **AWS CLI** configured with credentials that have AgentCore permissions

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import getpass
import json
import logging
import time
import uuid

import boto3
import requests
from boto3.session import Session

import gateway_mcp_client

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler()],
)

session = Session()
REGION = session.region_name or "us-east-1"
print(f"Using region: {REGION}")

In [ ]:
# Collect Salesforce credentials (never hardcoded)
SF_DOMAIN = input("Enter your Salesforce domain (e.g., myorg-dev-ed): ")
SF_CLIENT_ID = input("Enter your Salesforce Consumer Key (Client ID): ")
SF_CLIENT_SECRET = getpass.getpass("Enter your Salesforce Consumer Secret: ")

assert SF_DOMAIN.strip(), "Salesforce domain cannot be empty"
assert SF_CLIENT_ID.strip(), "Client ID cannot be empty"
assert SF_CLIENT_SECRET.strip(), "Client Secret cannot be empty"

print(f"\nSalesforce domain: {SF_DOMAIN}")
print(f"Token endpoint: https://{SF_DOMAIN}.develop.my.salesforce.com/services/oauth2/token")

## Step 1: Create a Salesforce Connected App

If you haven't already created a Connected App, follow these steps in your Salesforce org:

### Option A: Connected App (Classic)

1. Log in to Salesforce → Setup (gear icon → Setup)
2. Quick Find → **App Manager** → **New Connected App**
3. Enable OAuth Settings:
   - Callback URL: `https://bedrock-agentcore.<your-region>.amazonaws.com/identities/oauth2/callback`
   - OAuth Scopes: `Full access (full)`, `Perform requests at any time (refresh_token, offline_access)`, `Manage user data via APIs (api)`
4. Security settings:
   - ✅ Require Proof Key for Code Exchange (PKCE)
   - ✅ Require Secret for Web Server Flow
   - ✅ Enable Client Credentials Flow
5. Save → wait 2-10 minutes for propagation
6. **Critical:** App Manager → find your app → Manage → Edit Policies → Client Credentials Flow → set **Run As** to your admin username
7. Get Consumer Key + Consumer Secret from Manage Consumer Details

### Option B: External Client App (newer orgs)

1. Setup → **External Client App Manager** → New External Client App
2. Enable OAuth with same callback URL and scopes as above
3. Enable Client Credentials Flow with Run As user = admin username
4. Get Client ID and Secret from the app's OAuth settings page

> **Note:** Developer Edition orgs hibernate after ~24h of inactivity. If you get HTTP 420 errors, log into the Salesforce web UI to wake the org.

## Step 2: Verify Salesforce OAuth2 Token

Test that your credentials work before configuring the gateway.

In [ ]:
# Test Salesforce OAuth2 client_credentials flow
token_endpoint = f"https://{SF_DOMAIN}.develop.my.salesforce.com/services/oauth2/token"

resp = requests.post(
    token_endpoint,
    data={
        "grant_type": "client_credentials",
        "client_id": SF_CLIENT_ID,
        "client_secret": SF_CLIENT_SECRET,
    },
    headers={"Content-Type": "application/x-www-form-urlencoded"},
    timeout=30,
)

if resp.status_code == 200:
    sf_token_data = resp.json()
    print("✓ Salesforce OAuth2 token obtained successfully")
    print(f"  Token type: {sf_token_data.get('token_type')}")
    print(f"  Instance URL: {sf_token_data.get('instance_url')}")
else:
    print(f"✗ Failed to get token: {resp.status_code}")
    print(f"  Response: {resp.text}")
    raise RuntimeError("Fix your Salesforce credentials before proceeding")

## Step 3: Create AgentCore Gateway with Cognito Inbound Auth

We create a Cognito User Pool for gateway inbound authentication (machine-to-machine), then create the gateway.

In [ ]:
GATEWAY_NAME = f"multi-isv-tutorial-{str(uuid.uuid4())[:8]}"
print(f"Gateway name: {GATEWAY_NAME}")

# Create Cognito User Pool for gateway inbound auth
cognito_client = boto3.client("cognito-idp", region_name=REGION)

pool_resp = cognito_client.create_user_pool(
    PoolName=f"{GATEWAY_NAME}-pool",
    Policies={"PasswordPolicy": {"MinimumLength": 8}},
)
USER_POOL_ID = pool_resp["UserPool"]["Id"]
print(f"Created User Pool: {USER_POOL_ID}")

# Create domain for the pool
COGNITO_DOMAIN = f"{GATEWAY_NAME}-domain"
cognito_client.create_user_pool_domain(
    Domain=COGNITO_DOMAIN,
    UserPoolId=USER_POOL_ID,
)
TOKEN_ENDPOINT = f"https://{COGNITO_DOMAIN}.auth.{REGION}.amazoncognito.com/oauth2/token"
print(f"Token endpoint: {TOKEN_ENDPOINT}")

# Create resource server (defines the scope)
SCOPE_NAME = "invoke"
RESOURCE_SERVER_ID = f"{GATEWAY_NAME}-id"
cognito_client.create_resource_server(
    UserPoolId=USER_POOL_ID,
    Identifier=RESOURCE_SERVER_ID,
    Name=f"{GATEWAY_NAME} Gateway",
    Scopes=[{"ScopeName": SCOPE_NAME, "ScopeDescription": "Invoke gateway"}],
)
FULL_SCOPE = f"{RESOURCE_SERVER_ID}/{SCOPE_NAME}"
print(f"Scope: {FULL_SCOPE}")

# Create app client with client_credentials grant
app_resp = cognito_client.create_user_pool_client(
    UserPoolId=USER_POOL_ID,
    ClientName=f"{GATEWAY_NAME}-client",
    GenerateSecret=True,
    AllowedOAuthFlows=["client_credentials"],
    AllowedOAuthScopes=[FULL_SCOPE],
    AllowedOAuthFlowsUserPoolClient=True,
)
GW_CLIENT_ID = app_resp["UserPoolClient"]["ClientId"]
GW_CLIENT_SECRET = app_resp["UserPoolClient"]["ClientSecret"]
print(f"App client created: {GW_CLIENT_ID}")

DISCOVERY_URL = f"https://cognito-idp.{REGION}.amazonaws.com/{USER_POOL_ID}/.well-known/openid-configuration"
print(f"Discovery URL: {DISCOVERY_URL}")

In [ ]:
# Create IAM role for the gateway
iam = boto3.client("iam")
ROLE_NAME = f"agentcore-{GATEWAY_NAME}-role"

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "bedrock-agentcore.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}

role_resp = iam.create_role(
    RoleName=ROLE_NAME,
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="IAM role for AgentCore Gateway multi-ISV tutorial",
)
ROLE_ARN = role_resp["Role"]["Arn"]
print(f"Created IAM role: {ROLE_ARN}")

# Wait for IAM propagation
time.sleep(10)

In [ ]:
# Create the AgentCore Gateway
gateway_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

gw_resp = gateway_client.create_gateway(
    name=GATEWAY_NAME,
    roleArn=ROLE_ARN,
    protocolType="MCP",
    authorizerType="CUSTOM_JWT",
    authorizerConfiguration={
        "customJWTAuthorizer": {
            "discoveryUrl": DISCOVERY_URL,
            "allowedClients": [GW_CLIENT_ID],
        }
    },
)

GATEWAY_ID = gw_resp["gatewayId"]
GATEWAY_URL = gw_resp["gatewayUrl"]
print(f"Gateway created: {GATEWAY_ID}")
print(f"Gateway URL: {GATEWAY_URL}")

# Wait for gateway to become READY
print("Waiting for gateway to become READY...")
for _ in range(60):
    status = gateway_client.get_gateway(gatewayIdentifier=GATEWAY_ID)["status"]
    print(f"  Status: {status}")
    if status == "READY":
        break
    time.sleep(5)

## Step 4: Create CustomOauth2 Credential Provider

AgentCore Gateway needs OAuth2 credentials to authenticate to Salesforce on behalf of the agent.

> **Important:** Use `CustomOauth2` — NOT `SalesforceOauth2`. The built-in Salesforce vendor uses `login.salesforce.com` which does not support `client_credentials` on Developer Edition org domains. With `CustomOauth2`, we specify the org-specific token endpoint directly.

In [ ]:
CREDENTIAL_PROVIDER_NAME = f"{GATEWAY_NAME}-sf-oauth"

identity_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

cred_resp = identity_client.create_oauth2_credential_provider(
    name=CREDENTIAL_PROVIDER_NAME,
    credentialProviderVendor="CustomOauth2",
    oauth2ProviderConfigInput={
        "customOauth2ProviderConfig": {
            "clientId": SF_CLIENT_ID,
            "clientSecret": SF_CLIENT_SECRET,
            "oauthDiscovery": {
                "authorizationServerMetadata": {
                    "issuer": f"https://{SF_DOMAIN}.develop.my.salesforce.com",
                    "authorizationEndpoint": f"https://{SF_DOMAIN}.develop.my.salesforce.com/services/oauth2/authorize",
                    "tokenEndpoint": f"https://{SF_DOMAIN}.develop.my.salesforce.com/services/oauth2/token",
                }
            },
        }
    },
)

CREDENTIAL_PROVIDER_ARN = cred_resp["credentialProviderArn"]
print(f"Created credential provider: {CREDENTIAL_PROVIDER_NAME}")
print(f"ARN: {CREDENTIAL_PROVIDER_ARN}")

## Step 5: Add Salesforce as Gateway Target

Now we add Salesforce Lightning Platform as an integration target on the gateway. The gateway uses the pre-built Salesforce template to expose Salesforce REST APIs as MCP tools.

In [ ]:
SF_TARGET_NAME = "salesforce-target"
SF_SERVER_URL = f"https://{SF_DOMAIN}.develop.my.salesforce.com/services/data/v62.0"

target_resp = gateway_client.create_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    name=SF_TARGET_NAME,
    targetConfiguration={
        "mcp": {
            "openApiSchema": {
                "s3": {
                    "uri": "s3://amazonbedrockagentcore-built-sampleschemas455e0815-oj7jujcd8xiu/salesforce-open-api.json"
                }
            }
        }
    },
    credentialProviderConfigurations=[
        {
            "credentialProviderType": "OAUTH",
            "credentialProvider": {
                "oauthCredentialProvider": {
                    "providerArn": CREDENTIAL_PROVIDER_ARN,
                    "scopes": [],
                    "grantType": "CLIENT_CREDENTIALS",
                }
            },
        }
    ],
)

SF_TARGET_ID = target_resp["targetId"]
print(f"Created Salesforce target: {SF_TARGET_NAME} ({SF_TARGET_ID})")
print("Waiting for target to reach READY status...")

In [ ]:
# Wait for target to become READY
gateway_mcp_client.wait_for_target_ready(
    client=gateway_client,
    gateway_id=GATEWAY_ID,
    target_name=SF_TARGET_NAME,
    region=REGION,
    timeout=300,
)
print("\n✓ Salesforce target is READY")

## Step 6: List Salesforce Tools via Gateway

Now that the target is ready, we can list all available Salesforce tools through the gateway's MCP endpoint.

In [ ]:
# Get a gateway access token
def get_gw_token() -> str:
    return gateway_mcp_client.get_cognito_m2m_token(
        token_endpoint=TOKEN_ENDPOINT,
        client_id=GW_CLIENT_ID,
        client_secret=GW_CLIENT_SECRET,
        scope=FULL_SCOPE,
    )

mcp = gateway_mcp_client.GatewayMCPClient(
    gateway_url=GATEWAY_URL,
    get_token=get_gw_token,
    session_id=str(uuid.uuid4()),
)

# List all tools (handles pagination automatically)
all_tools = mcp.list_all_tools()
sf_tools = [t for t in all_tools if t["name"].startswith("salesforce-target___")]

print(f"Total tools on gateway: {len(all_tools)}")
print(f"Salesforce tools: {len(sf_tools)}")
print("\nSalesforce tool names:")
for t in sorted(sf_tools, key=lambda x: x["name"]):
    print(f"  - {t['name']}")

## Step 7: Invoke Salesforce Tools

Let's call some Salesforce tools through the gateway.

> **Important:** Every Salesforce tool call requires the `domainName` parameter. Without it, the gateway resolves to a non-existent default domain.

In [ ]:
# Query accounts using SOQL (more reliable than getAccountList)
result = mcp.call_tool(
    "salesforce-target___queryAccounts",
    {"domainName": SF_DOMAIN, "q": "SELECT Id, Name, Industry FROM Account LIMIT 5"},
)
print("=== Query Accounts (SOQL) ===")
print(json.dumps(result.get("result", {}), indent=2)[:2000])

In [ ]:
# Get account list (returns recently viewed accounts)
result = mcp.call_tool(
    "salesforce-target___getAccountList",
    {"domainName": SF_DOMAIN},
)
print("=== Get Account List ===")
print(json.dumps(result.get("result", {}), indent=2)[:2000])

In [ ]:
# Create a test account
# Note: pass Content-Type as empty string to work around a known header duplication issue
result = mcp.call_tool(
    "salesforce-target___createAccount",
    {
        "domainName": SF_DOMAIN,
        "Content-Type": "",
        "Name": "AgentCore Tutorial Test Account",
        "Industry": "Technology",
        "Description": "Created by AgentCore Gateway multi-ISV tutorial",
    },
)
print("=== Create Account ===")
print(json.dumps(result.get("result", {}), indent=2)[:2000])

In [ ]:
# Describe an SObject to see available fields
result = mcp.call_tool(
    "salesforce-target___describeSObject",
    {"domainName": SF_DOMAIN, "sObjectType": "Account"},
)
print("=== Describe Account SObject ===")
content = result.get("result", {}).get("content", [])
if content:
    text = content[0].get("text", "")
    try:
        data = json.loads(text)
        fields = data.get("fields", [])[:10]
        print(f"Total fields: {len(data.get('fields', []))}")
        print("First 10 fields:")
        for f in fields:
            print(f"  - {f['name']} ({f['type']})")
    except json.JSONDecodeError:
        print(text[:1000])

## Known Issues & Workarounds

| Issue | Workaround |
|---|---|
| `create*` tools fail with HTTP 415 | Pass `"Content-Type": ""` (empty string) in tool arguments |
| Tool calls return HTTP 420 | Always include `domainName` in arguments; also check if org is hibernating |
| `getAccountList` returns only recently viewed | Use `queryAccounts` with SOQL: `SELECT Id, Name FROM Account` |
| `SalesforceOauth2` vendor fails with Dev orgs | Use `CustomOauth2` with org-specific token endpoint (as done above) |
| Org hibernation (HTTP 420 after inactivity) | Log into Salesforce web UI to wake the org |

## Step 8: Use Strands Agent with Salesforce Tools

Now let's connect a Strands Agent to the gateway and let it use Salesforce tools via natural language.

In [ ]:
from strands import Agent
from strands.tools.mcp import MCPClient
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

# Connect to gateway via MCP
mcp_client = MCPClient(
    lambda: streamablehttp_client(
        url=GATEWAY_URL,
        headers={
            "Authorization": f"Bearer {get_gw_token()}",
            "MCP-Protocol-Version": "2025-03-26",
        },
    )
)

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to Salesforce tools. "
    f"Always include domainName='{SF_DOMAIN}' in every Salesforce tool call. "
    "Do not pass a Content-Type parameter to Salesforce create operations — omit it entirely or pass empty string. "
    "Use queryAccounts with SOQL for listing accounts rather than getAccountList."
)

with mcp_client:
    agent = Agent(
        model="us.anthropic.claude-sonnet-4-6-v1",
        system_prompt=SYSTEM_PROMPT,
        tools=mcp_client.list_tools_sync(),
    )
    result = agent("List all Salesforce accounts with their industry")
    print(result)

## Step 9: Clean Up

Delete all resources created in this notebook.

In [ ]:
# Delete gateway target
print("Deleting Salesforce gateway target...")
gateway_client.delete_gateway_target(
    gatewayIdentifier=GATEWAY_ID,
    targetId=SF_TARGET_ID,
)
print("  ✓ Target deleted")

# Delete credential provider
print("Deleting credential provider...")
identity_client.delete_oauth2_credential_provider(name=CREDENTIAL_PROVIDER_NAME)
print("  ✓ Credential provider deleted")

# Delete gateway
print("Deleting gateway...")
gateway_client.delete_gateway(gatewayIdentifier=GATEWAY_ID)
print("  ✓ Gateway deleted")

# Delete Cognito resources
print("Deleting Cognito resources...")
cognito_client.delete_user_pool_domain(Domain=COGNITO_DOMAIN, UserPoolId=USER_POOL_ID)
cognito_client.delete_user_pool(UserPoolId=USER_POOL_ID)
print("  ✓ Cognito pool deleted")

# Delete IAM role
print("Deleting IAM role...")
iam.delete_role(RoleName=ROLE_NAME)
print("  ✓ IAM role deleted")

print("\n✓ All resources cleaned up")